# Ablation: LN Removal vs. GITA Gamma Correction

Reviewer comment: *"Layer normalization could potentially be replaced by the proposed gamma-correction-based input transformation."*

| Scenario | LN Removed | LN Kept | TTA |
|----------|-----------|---------|-----|
| **C** | layers 2-3, PatchMerging, norm0-3, patch_embed | layers.[01].blocks.*.norm[12] | None (direct test) |
| **D** | Same as C | Same (inside CascadeAnchor) | GITA (gamma ITM) |

The only difference between C and D is **whether GITA is applied**. The set of removed LN layers is identical.

In [ ]:
import re
from os import path, makedirs, environ, system
import json

BATCH_SIZE     = 1
FIT_BATCH_SIZE = 40
TOTAL_ROUNDS   = 1
DATA_ROOT      = path.join(".", "data")
RESULT_ROOT    = path.join(".", "results-rebuttal")
DEVICE_NUM     = 0

# Matches the LN layers that GITA targets (kept in both C and D for fair comparison)
GITA_SWIN_PATTERN = r"\.layers\.[01]\.blocks\..*\.norm[12]$"

In [ ]:
_ = system("nvidia-smi")

In [ ]:
import torch
import torch.nn as nn

environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE_NUM)
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"INFO: Using device - {device}:{DEVICE_NUM}")

In [ ]:
from ttadapters import datasets, models, methods
from ttadapters.datasets import scenarios
from ttadapters.utils.validator import DetectionEvaluator
from ttadapters.utils.visualizer import visualize_metrics
from ttadapters.methods.auto import CONFIG_MAPPING

import pandas as pd
pd.options.display.float_format = lambda x: f"{x*100 if x < 1 else x:.4f}"

In [ ]:
datasets.patch_fast_download_for_object_detection()

train_dataset = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT, train=True)
CLASSES = train_dataset.classes
print(f"INFO: Number of classes - {len(CLASSES)} {CLASSES}")

In [ ]:
def remove_layer_norm(model: nn.Module, exclude_pattern: str = None) -> int:
    """Replace LayerNorm layers with nn.Identity.

    Args:
        exclude_pattern: regex. Matching layer names are KEPT. None = replace all.

    Returns:
        Number of replaced layers.
    """
    exc_re = re.compile(exclude_pattern) if exclude_pattern else None
    to_replace = []

    for name, module in model.named_modules():
        is_ln = isinstance(module, nn.LayerNorm) or "LayerNorm" in module.__class__.__name__
        if is_ln and (exc_re is None or not exc_re.search(name)):
            to_replace.append(name)

    for name in to_replace:
        parts = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], nn.Identity())

    return len(to_replace)


def make_json_serializable(obj):
    if isinstance(obj, list):
        return [make_json_serializable(i) for i in obj]
    if isinstance(obj, dict):
        return {(k.value if hasattr(k, "value") else k): make_json_serializable(v) for k, v in obj.items()}
    return obj


result_dir = path.join(RESULT_ROOT, "swinrcnn", "continual_tta", "ln_removal_ablation")
makedirs(result_dir, exist_ok=True)

---
## Scenario C — Partial LN Removed, Direct Test

Replace all non-GITA-target LayerNorm with Identity, then run inference without any adaptation.  
Measures the performance drop from LN removal as a baseline.

In [ ]:
base_model_c, load_result_c = models.SwinRCNNForObjectDetection.from_dataset(dataset=datasets.SHIFTDataset)
data_preparation = base_model_c.DataPreparation(train_dataset, evaluation_mode=True)
print("INFO: Model state loaded -", load_result_c)

# Remove same LN set as D (exclude GITA targets) for fair comparison
n_c = remove_layer_norm(base_model_c, exclude_pattern=GITA_SWIN_PATTERN)
print(f"C: Replaced {n_c} LayerNorm -> Identity (layers.[01].blocks.*.norm[12] kept)")

base_model_c.to(device)
base_model_c.eval()

scenario_params = dict(root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None)
loader_params   = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
tta_c = methods.MethodContainer(**{"C_SwinRCNN_NoLN": base_model_c})

evaluator_c = DetectionEvaluator(
    tta_c.methods(), classes=CLASSES,
    data_preparation=data_preparation,
    dtype=torch.float32, device=device,
    no_grad=True
)

result_c = []
for rd, this in tta_c.go_rounds(end_round=TOTAL_ROUNDS):
    continual_scenario_c = scenarios.SHIFTDiscreteScenarioForContinualTTA(**scenario_params)
    result = visualize_metrics(continual_scenario_c(**loader_params).play(evaluator_c, index=this))
    result_c.append(result)
    with open(path.join(result_dir, f"C_result_r{rd}_b{BATCH_SIZE}.json"), "w", encoding="utf-8") as f:
        json.dump(make_json_serializable(result), f)

print(f"\nC done. Saved to: {result_dir}")

---
## Scenario D — Partial LN Removed + GITA

Apply GITA first so that target LN layers are wrapped with CascadeAnchor, then remove all remaining LN.  
Since CascadeAnchor does not contain "LayerNorm" in its class name, it is naturally excluded from removal.

**Removed**: layers 2-3 block norms, PatchMerging norms, stage output norms (norm0-3), patch_embed.norm  
**Kept (inside CascadeAnchor)**: `layers.[01].blocks.*.norm[12]` — still performs normalization and provides anchor signal for gamma adaptation

In [ ]:
base_model_d, load_result_d = models.SwinRCNNForObjectDetection.from_dataset(dataset=datasets.SHIFTDataset)
print("INFO: Model state loaded -", load_result_d)

In [ ]:
config_d = CONFIG_MAPPING["gita_engine"].from_preset(base_model_d)
adaptive_model_d = methods.AutoAdaptationEngineForObjectDetection.from_config(config_d, base_model=base_model_d)
adaptive_model_d.to(device)

# Remove remaining LN after GITA wraps its targets with CascadeAnchor.
# CascadeAnchor class name has no "LayerNorm" -> naturally excluded from removal.
n_d = remove_layer_norm(adaptive_model_d.base_model)
print(f"D: Replaced {n_d} LayerNorm -> Identity (anchor layers protected by CascadeAnchor)")

In [ ]:
adaptive_model_d.fit(data_preparation, batch_size=FIT_BATCH_SIZE, shuffle=False)

In [ ]:
adaptive_model_d.online()

tta_d = methods.MethodContainer(**{
    f"D_SwinRCNN_PartialLN_{adaptive_model_d.model_type}": adaptive_model_d
})

evaluator_d = DetectionEvaluator(
    tta_d.methods(), classes=CLASSES,
    data_preparation=data_preparation,
    dtype=torch.float32, device=device,
    no_grad=False
)

result_d = []
for rd, this in tta_d.go_rounds(end_round=TOTAL_ROUNDS):
    continual_scenario_d = scenarios.SHIFTDiscreteScenarioForContinualTTA(**scenario_params)
    result = visualize_metrics(continual_scenario_d(**loader_params).play(evaluator_d, index=this))
    result_d.append(result)
    with open(path.join(result_dir, f"D_result_r{rd}_b{BATCH_SIZE}.json"), "w", encoding="utf-8") as f:
        json.dump(make_json_serializable(result), f)

print(f"\nD done. Saved to: {result_dir}")